##  First : 算法运行总脚本
运行 ......文件夹下的 xxxxx.sh 既可以得到实验绘图所需的所有数据

## Second : 算法流水线
按照下面步骤一步步运行得到论文 Evaluation 所需的所有方法的输出结果

### A. 计算 Ground_truth (EXACT)
(1)  运行 `exact_subgraph_match.py`---> 调用 CProject 下 精确子图匹配算法 产生精确子图匹配结果
(2)  运行 python `EXACT.py` 文件 用查询对应的oracle谓词验证精确子图匹配结果 ,agg_mode={count,sum}

### B. \text{PROXY} count/sum 实验

1. 运行 `Projection_Sampling_and_Weight_Estimation_Runner.py` 得到论文中所提的 $\hat{\Psi}$ 和 $\hat{w}(\psi)$。

2. **RQ1 & RQ2 & RQ4**: 运行 `Proxy_Guided_Stratified_Importance_Sampling_Runner.py` 在 $\hat{\Psi}$ 上进行分层重要性采样：
   
   * **a. 对于 RQ1 & RQ2：**
     ```python
     methods_map = {
         "PROXY": sampler.run_possa,
     }
     --target_ticks 0.01,0.05,0.075,0.1,0.125,0.15,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9
     ```
    结果保存到: `allocation_strategy_comparison_{agg_mode}.csv`
   * **b. 对于 RQ4：**
     ```python
     methods_map = {
         "PROJ": sampler.run_baseline_uniform,
         "PO": sampler.run_baseline_proxy,
         "WO": sampler.run_baseline_weight_only,
         "MAB": sampler.run_mab_sampling,
         "PROXY": sampler.run_possa,
     }
     --target_ticks 0.1
     ```
    结果保存到: `allocation_strategy_comparison_ablation_{agg_mode}.csv`


3. **RQ3**: 运行 `Sensitivity_single_predicate_Runner.py` 和 `Sensitivity_multi_predicate_comparation.py` 分别检验面对多谓词和单谓词的退化，$\text{PROXY}$ 的鲁棒性。
    结果保存到: `proxy_quality_ablation_{aggmode}.csv`

### C. $\text{PROXY}$ avg 实验

$\text{AVG}$ 估计基于定理 6 的比率估计器。该实验不运行 C++ 引擎，而是通过离线合并已完成的 `count` 和 `sum` 实验结果进行数据合成：

1. **Ground Truth 合并**：通过 $\tau_{\operatorname{avg}} = \tau_{\operatorname{sum}} / \tau_{\operatorname{count}}$ 计算真值。
   * 输入：`T_true_*_sum.json` 与 `T_true_*_count.json`
   * 结果保存到：`T_true_*_avg.json`

2. **RQ1 & RQ2 & RQ4**：运行比率对齐脚本，按照 $\hat{\tau}_{\operatorname{avg}} = \hat{\tau}_{\operatorname{sum}} / \hat{\tau}_{\operatorname{count}}$ 合成估计值并计算误差：
   * **a. 核心与消融策略（RQ1, RQ2 & RQ4）**：合并 `allocation_strategy_comparison_{count,sum}.csv` $\rightarrow$ 保存到 `allocation_strategy_comparison_avg.csv`
   * **b. Fastest-Oracle**：合并 `FastestO_budget_curve_{count,sum}.csv` $\rightarrow$ 保存到 `FastestO_budget_curve_avg.csv`
   * **c. 精确匹配基准 Exact\_structureO**：合并 `Exact_structureO_budget_curve_{count,sum}.csv` $\rightarrow$ 保存到 `Exact_structureO_budget_curve_avg.csv`

3. **自动兼容**：脚本自动检测并适配以下采样统计列：
   * **Parler 风格**：自动提取 `n_post` 和 `n_comment`
   * **Amazon 风格**：自动提取 `n_product` 和 `n_review`

### D. 按照时间对等协议计算 $B_{virtual}$
1. 读取 allocation_strategy_comparison_{count,sum}.csv 保存每一个 oracle 模型 和 proxy 模型在 $\hat{\Psi}$ 空间采样率 $\alpha$ 下的运行次数 $N_{oi}$ 和 $N_{pi}$.

2. 根据 在每个数据集测试的 oracle模型 和 proxy 模型的平均延迟, $c_i$ 和 $c_p^i$ 计算 $B_{virtual}$

### E 根据  $B_{virtual}$ 计算 ENUM, FASTEST-ORACLE 估计 $\hat{\tau}$ 的 AAE 


####  计算 ENUM 结果
(1) 运行 `ENUM.py` 文件 , 计算ENUM方法在 workload 下, agg_mode={count,sum}


####  计算Fastest_Oracle 结果
(1) 运行 `FASTEST-ORACLE.py` 文件  ---> 调用 CProject 中的 FASTEST-Oracle 算法估计 $\hat{\tau}$ ,agg_mode={count,sum}

文件输出POSS的结果到/results/efficiency 文件夹下的 `FastestO_budget_curve_{agg_mode}.csv` 文件中,
该文件记录了每个查询 $Q$ 在 数据集=dataset_name, 采样率为 $alpha$ 下, 运行 k=5 or 10 次的 $\hat{\tau}_{proxy}^Q$ 

#### F. 计算WEE 渐近线
(1)  运行 `WEE.py` 文件得到每个 dataset 的 WEE